## Working with Maidenhead in Vgrid DGGS

[![image](https://jupyterlite.rtfd.io/en/latest/_static/badge.svg)](https://demo.vgrid.vn/lab/index.html?path=vgrid/17_maidenhead.ipynb)
[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeoshub/vgrid/blob/main/docs/notebooks/17_maidenhead.ipynb)
[![image](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/opengeoshub/vgrid/HEAD?filepath=docs/notebooks/17_maidenhead.ipynb)
[![image](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/opengeoshub/vgrid/blob/main/docs/notebooks/17_maidenhead.ipynb)
[![image](https://jupyterlite.rtfd.io/en/latest/_static/badge.svg)](https://jupyterlite.gishub.vn/lab/index.html?path=notebooks/vgrid/17_maidenhead.ipynb)

Full Vgrid DGGS documentation is available at [vgrid document](https://vgrid.gishub.vn).

To work with Vgrid DGGS directly in GeoPandas and Pandas, please use [vgridpandas](https://pypi.org/project/vgridpandas/). Full Vgridpandas DGGS documentation is available at [vgridpandas document](https://vgridpandas.gishub.vn).

To work with Vgrid DGGS in QGIS, install the [Vgrid Plugin](https://plugins.qgis.org/plugins/vgridtools/).

To visualize DGGS in Maplibre GL JS, try the [vgrid-maplibre](https://www.npmjs.com/package/vgrid-maplibre) library.

For an interactive demo, visit the [Vgrid Homepage](https://vgrid.vn).

### Install vgrid
Uncomment the following line to install [vgrid](https://pypi.org/project/vgrid/).

In [ ]:
# %pip install vgrid --upgrade

### latlon2maidenhead

In [ ]:
from vgrid.conversion.latlon2dggs import latlon2maidenhead

lat = 10.775276
lon = 106.706797
res = 4
maidenhead_id = latlon2maidenhead(lat, lon, res)
maidenhead_id

### Maidenhead to Polygon

In [ ]:
from vgrid.conversion.dggs2geo.maidenhead2geo import maidenhead2geo

maidenhead_geo = maidenhead2geo(maidenhead_id)
maidenhead_geo

### Maidenhead to GeoJSON

In [ ]:
from vgrid.conversion.dggs2geo.maidenhead2geo import maidenhead2geojson

maidenhead_geojson = maidenhead2geojson(maidenhead_id)
# maidenhead_geojson

### Vector to Maindenhead

In [ ]:
from vgrid.conversion.vector2dggs.vector2maidenhead import vector2maidenhead

file_path = "https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/polygon.geojson"
vector_to_maidenhead = vector2maidenhead(
    file_path,
    resolution=4,
    compact=False,
    topology=False,
    predicate="intersects",
    output_format="gpd",
)
# Visualize the output
# vector_to_georef
vector_to_maidenhead.plot(edgecolor="white")

### Maidenhead Binning

In [ ]:
from vgrid.binning.maidenheadbin import maidenheadbin

file_path = (
    "https://raw.githubusercontent.com/opengeoshub/vopendata/main/csv/housing.csv"
)
stats = "max"
maidenhead_bin = maidenheadbin(
    file_path,
    resolution=3,
    stats=stats,
    numeric_col="median_house_value",
    # category_col="category",
    output_format="gpd",
)
maidenhead_bin.plot(
    column=stats,  # numeric column to base the colors on
    cmap="Spectral_r",  # color scheme (matplotlib colormap)
    legend=True,
    linewidth=0.2,  # boundary width (optional)
)

### Raster to Maidenhead

#### Download and open raster

In [ ]:
from vgrid.utils.io import download_file
import rasterio
from rasterio.plot import show

raster_url = (
    "https://raw.githubusercontent.com/opengeoshub/vopendata/main/raster/rgb.tif"
)
raster_file = download_file(raster_url)
src = rasterio.open(raster_file, "r")
print(src.meta)
show(src)

#### Convert raster to GARS

In [ ]:
# %pip install folium

In [ ]:
from vgrid.conversion.raster2dggs.raster2maidenhead import raster2maidenhead

raster_to_maidenhead = raster2maidenhead(raster_file, resolution = 2, stats = 'mean', output_format="gpd")

# Visualize the output
import folium

m = folium.Map(tiles="CartoDB positron", max_zoom=28)

maidenhead_layer = folium.GeoJson(
    raster_to_maidenhead,
    style_function=lambda x: {
        "fillColor": f"rgb({x['properties']['band_1']}, {x['properties']['band_2']}, {x['properties']['band_3']})",
        "fillOpacity": 1,
        "color": "black",
        "weight": 1,
    },
    popup=folium.GeoJsonPopup(
        fields=["maidenhead", "band_1", "band_2", "band_3"],
        aliases=["Maidenhead ID", "Band 1", "Band 2", "Band 3"],
        style="""
            background-color: white;
            border: 2px solid black;
            border-radius: 3px;
            box-shadow: 3px;
        """,
    ),
).add_to(m)

m.fit_bounds(maidenhead_layer.get_bounds())

# Display the map
m

### Maidenhead Generator

In [ ]:
from vgrid.generator.maidenheadgrid import maidenheadgrid

maidenhead_grid = maidenheadgrid(resolution=2, output_format="gpd")
# maidenhead_grid = maidenheadgrid(resolution=3,bbox=[102.14,7.69,114.86,23.39],output_format="gpd")
maidenhead_grid.plot(edgecolor="#3474eb", facecolor="none", linewidth=0.1)

### Maidenhead Inspect

In [ ]:
from vgrid.stats.maidenheadstats import maidenheadinspect

resolution = 2
maidenhead_inspect = maidenheadinspect(resolution)
maidenhead_inspect.head()

### Maidenhead Normalized Area Histogram

In [ ]:
from vgrid.stats.maidenheadstats import maidenhead_norm_area_hist

maidenhead_norm_area_hist(maidenhead_inspect)

### Distribution of Maidenhead Area Distortions

In [ ]:
from vgrid.stats.maidenheadstats import maidenhead_norm_area

maidenhead_norm_area(maidenhead_inspect)

### Maidenhead IPQ Compactness Histogram

Isoperimetric Inequality (IPQ) Compactness (suggested by [Osserman, 1978](https://sites.math.washington.edu/~toro/Courses/20-21/MSF/osserman.pdf)):

$$C_{IPQ} = \frac{4 \pi A}{p^2}$$
The range of the IPQ compactness metric is [0,1]. 

A circle represents the maximum compactness with a value of 1. 

As shapes become more irregular or elongated, their compactness decreases toward 0.

In [ ]:
from vgrid.stats.maidenheadstats import maidenhead_compactness_ipq_hist

maidenhead_compactness_ipq_hist(maidenhead_inspect)

### Distribution of Maidenhead IPQ Compactness

In [ ]:
from vgrid.stats.maidenheadstats import maidenhead_compactness_ipq

maidenhead_compactness_ipq(maidenhead_inspect)

### Maidenhead Convex hull Compactness Histogram:

$$C_{CVH} = \frac{A}{A_{CVH}}$$


The range of the convex hull compactness metric is [0,1]. 

As shapes become more concave, their convex hull compactness decreases toward 0.

In [ ]:
from vgrid.stats.maidenheadstats import maidenhead_compactness_cvh_hist

maidenhead_compactness_cvh_hist(maidenhead_inspect)

### Distribution of Maidenhaed Convex hull Compactness

In [ ]:
from vgrid.stats.maidenheadstats import maidenhead_compactness_cvh

maidenhead_compactness_cvh(maidenhead_inspect)

### Maidenhead Statistics

Characteristic Length Scale (CLS - suggested by Ralph Kahn): the diameter of a spherical cap of the same cell's area

In [ ]:
from vgrid.stats import maidenheadstats

maidenheadstats("km")